# 📊 Automated Infrastructure Analysis with AI Agents

## 🧠 Introduction

This project implemented an artificial intelligence agent system to analyze structured JSON data on the status of an infrastructure composed of 100 servers. The goal was to simulate an environment where collaborative agents process, synthesize, and generate automated technical reports—just as a real-time monitoring team would.

The data includes key indicators such as CPU usage, memory usage, overall server status, and alerts triggered by critical events. To facilitate processing, this data was converted into a pandas `DataFrame`, allowing for structured analysis before being passed to the agent.

## 🎯 Learning Objectives

- Understand how to structure agents in CrewAI to collaborate in generating technical content.
- Use structured data (JSON) and transform it into a `DataFrame` to facilitate computational analysis.
- Automatically identify and classify critical servers or those with excessive resource usage.
- Generate Markdown reports that can be understood by both technical and executive audiences.
- Simulate a workflow where an analyst and a technical writer collaborate, reflecting a real-world professional dynamic.
- Explore the real capabilities and limitations of LLM agents when dealing with raw data.

This project demonstrates how modern AI tools can automate technical analysis and documentation tasks, delivering immediate value in contexts such as infrastructure, monitoring, and decision-making.


In [ ]:
!pip install crewai crewai-tools openai --quiet

In [ ]:
pip install crewai crewai-tools

In [ ]:
# === 📁 CARGAR ARCHIVO JSON DESDE COLAB ===
from google.colab import files
import json
import os
from getpass import getpass

from crewai import Agent, Task, Crew

# === 🔐 API KEY & MODELO GPT-3.5 ===
api_key = getpass("🔑 Ingresa tu API key de OpenAI: ").strip()
os.environ["OPENAI_API_KEY"] = api_key
os.environ["CREWAI_OPENAI_MODEL"] = "gpt-3.5-turbo"


In [ ]:

# === 📤 SUBIR ARCHIVO JSON DE SERVIDORES ===
uploaded = files.upload()
json_filename = list(uploaded.keys())[0]

with open(json_filename, "r") as f:
    servidores_data = json.load(f)

# === 📝 PREVISUALIZACIÓN DE LOS DATOS ===
resumen_entrada = f"Total servidores: {len(servidores_data)}\n"
resumen_entrada += json.dumps(servidores_data[:5], indent=2) + "\n... (muestra parcial de 100 registros)"


In [ ]:

# === 🤖 AGENTES ===

infra_analyst = Agent(
    role='Analista de Infraestructura',
    goal='Detectar servidores en riesgo a partir del JSON',
    backstory='Ingeniero de operaciones que analiza infraestructura crítica para TI.',
    tools=[],
    verbose=True
)

reporter = Agent(
    role='Redactor Técnico',
    goal='Convertir hallazgos técnicos en un informe comprensible',
    backstory='Redactor técnico con experiencia explicando problemas complejos a gerentes.',
    tools=[],
    verbose=True
)

analyze_task = Task(
    description=(
        "Este es el informe técnico generado a partir del análisis de los datos de infraestructura de 100 servidores. "
        "Todos los hallazgos están presentados directamente a continuación, usando formato Markdown profesional.\n\n"
        "## Resumen General\n"
        "...\n\n"
        "## Servidores Críticos\n"
        "...\n\n"
        "## Uso Excesivo de Recursos\n"
        "...\n\n"
        "## Alertas Más Comunes\n"
        "...\n\n"
        "## Recomendaciones\n"
        "...\n\n"
        "Sustituye los puntos suspensivos con datos reales basados en los registros. No expliques, no des instrucciones. "
        "Simplemente entrega el contenido final, como si ya hubieras hecho el análisis."
    ),
    expected_output="Un informe técnico en formato Markdown estructurado, directo y profesional.",
    agent=infra_analyst
)

# === 📝 TAREA 2: REDACCIÓN DEL INFORME EJECUTIVO ===

report_task = Task(
    description=(
        "Con base en el análisis anterior, redacta un informe técnico ejecutivo en 3 párrafos usando formato Markdown.\n"
        "- Evita repeticiones.\n"
        "- Resume hallazgos clave y riesgos principales.\n"
        "- Usa lenguaje comprensible para un gerente de TI no técnico.\n"
        "- Termina con una recomendación de acción clara."
    ),
    expected_output="Un informe limpio y profesional con contexto, riesgos clave y acciones recomendadas.",
    agent=reporter,
    output_file="informe_servidores.md"
)

# === 🧩 CREACIÓN Y EJECUCIÓN DEL CREW ===

crew = Crew(
    agents=[infra_analyst, reporter],
    tasks=[analyze_task, report_task],
    verbose=True,
    planning=True
)

crew.kickoff()


## ⚠️ Limitation of GPT + CrewAI When Analyzing Raw JSON

### 🧠 Why Does It Still Fail?

Even when the prompt says “replace the ellipses with actual data,” **GPT lacks effective capability to analyze complex raw JSON structures** when they are injected directly into a prompt. Instead of making inferences from the data, the model falls back on pretrained patterns and responds as if solving a typical analysis exercise, listing steps or procedures.

---

### 🔍 Structural Limitation in CrewAI + GPT

This is not just a matter of how the prompt is written, but a **structural limitation of the CrewAI + GPT stack**:

- CrewAI **does not process or transform the data before passing it to the model**.
- GPT, when receiving a plain text block with JSON structure, **does not interpret or apply real logic to it**.
- Instead of analyzing the data, it **simulates analysis** using learned text patterns (“1. Collect data”, “2. Filter servers...”), never actually reaching the expected final result.

---

### ✅ Conclusion

To obtain useful responses, **the data must be pre-analyzed using Python** or another structured language, and then a **processed summary** should be presented to the model. Only then can GPT focus on generating narrative content, conclusions, or professional writing — not structural analysis tasks that exceed its capabilities without auxiliary tools.


In [ ]:
# === 📤 CARGAR JSON Y TRANSFORMAR A DATAFRAME ===
from google.colab import files
import pandas as pd
import json

uploaded = files.upload()
json_filename = list(uploaded.keys())[0]

with open(json_filename, "r") as f:
    servidores_alertas_100 = json.load(f)

# Convertir a DataFrame
df_servidores = pd.DataFrame(servidores_alertas_100)
df_servidores.head()


In [ ]:
# === DEFINIR AGENTE EN FORMATO CREWAI ===
from crewai import Agent

infra_agent = Agent(
    role="Infrastructure Analyst",
    goal="Generar un informe técnico basado en los datos de rendimiento y alertas de 100 servidores",
    backstory=(
        "Eres un analista especializado en monitoreo de infraestructura TI. "
        "Tu objetivo es evaluar el estado general de los servidores, identificar "
        "riesgos como uso excesivo de recursos y alertas frecuentes, y entregar un "
        "informe estructurado en Markdown. "
        "Tu análisis permite a los equipos técnicos tomar decisiones informadas y priorizar acciones."
    ),
    allow_delegation=False,
    verbose=True
)


In [ ]:
# === PROCESAR LOS DATOS Y CREAR INFORME ===
from collections import Counter

total_servidores = len(df_servidores)
conteo_estados = df_servidores['status'].value_counts().to_dict()

servidores_criticos = df_servidores[df_servidores['status'] == 'critical']['server_id'].tolist()

uso_excesivo_df = df_servidores[(df_servidores['cpu_usage'] > 90) | (df_servidores['memory_usage'] > 90)]
uso_excesivo = [
    f"{row['server_id']} (CPU {row['cpu_usage']}% | MEM {row['memory_usage']}%)"
    for _, row in uso_excesivo_df.iterrows()
]

todas_alertas = sum(df_servidores['alerts'].tolist(), [])
alertas_comunes = Counter(todas_alertas).most_common(5)

# Crear informe Markdown
informe = "## Resumen General\n"
informe += f"- Total de servidores: {total_servidores}\n"
for estado, cantidad in conteo_estados.items():
    informe += f"- {estado}: {cantidad}\n"

informe += "\n## Servidores Críticos\n"
informe += "\n".join(f"- {srv}" for srv in servidores_criticos)

informe += "\n\n## Uso Excesivo de Recursos\n"
informe += "\n".join(f"- {info}" for info in uso_excesivo)

informe += "\n\n## Alertas Más Comunes\n"
for i, (alerta, freq) in enumerate(alertas_comunes, 1):
    informe += f"{i}. {alerta} - {freq} veces\n"

informe += "\n## Recomendaciones\n"
informe += "- Monitorear servidores con uso >90%\n"
informe += "- Escalar soporte en críticos frecuentes\n"
informe += "- Automatizar respuestas ante alertas comunes\n"
informe += "- Revisar hardware en servidores con fallas recurrentes\n"
informe += "- Optimizar configuraciones en warning persistentes\n"

# Guardar informe
with open("informe_servidores.md", "w") as f:
    f.write(informe)


In [ ]:
print(informe)

In [ ]:
# Paso 3: Definir el agente y tarea con CrewAI
from crewai import Agent, Task, Crew

# El informe generado desde el análisis con DataFrame
contexto_datos = informe  # variable generada anteriormente

# Agente que redactará el informe final
redactor = Agent(
    role="Redactor Técnico",
    goal="Redactar un informe profesional claro y directo basado en datos técnicos procesados",
    backstory=(
        "Eres un redactor técnico con experiencia en infraestructura. "
        "Te han entregado un análisis ya realizado y tu tarea es transformar esos datos "
        "en un documento bien redactado, en tono ejecutivo, listo para gerencia."
    ),
    allow_delegation=False,
    verbose=True
)

# Tarea para que el agente redacte usando los datos ya analizados
tarea = Task(
    description=(
        "Con base en el siguiente informe técnico estructurado en Markdown, redacta un documento final claro, "
        "usando lenguaje profesional, sin repeticiones ni explicaciones técnicas. "
        "Conserva los encabezados, mejora la redacción y asegúrate de que el contenido sea comprensible por un gerente de TI.\n\n"
        f"{contexto_datos}"
    ),
    expected_output="Informe final en Markdown, redactado de forma clara y profesional.",
    agent=redactor,
    output_file="informe_final_redactado.md"
)

# Armar el crew y ejecutarlo
crew = Crew(
    agents=[redactor],
    tasks=[tarea],
    verbose=True,
    planning=True
)

crew.kickoff()
